In [1]:
### YOLOv8 Colab 학습 전체 코드
# 1. Ultralytics 설치
!pip install ultralytics

### 3. 이제 crop 으로 가자

In [2]:
import os
import shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import cv2
from ultralytics import YOLO

# ========================= [USER SETTINGS] ========================= #
# 1. 드라이브 내 원본 폴더 경로 (이미지가 들어있는 곳)
DRIVE_INPUT_DIR = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/0. 원본/251213-251226"

# 2. 결과물을 저장하고 싶은 드라이브 경로
DRIVE_OUTPUT_ROOT = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/2. ROI_BOX/251213-251226"

# 3. 모델 파일 경로
MODEL_PATH = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/1. ROI_JSON/ROI_detect/results/lettuce_yolov8n3/weights/best.pt"

# 기타 설정
NUM_WORKERS = 4
RESIZE_TO_256 = True
# =================================================================== #

In [3]:
# 로컬 작업용 경로 설정 (수정 금지)
LOCAL_INPUT_ZIP = "/content/temp_input.zip"
LOCAL_INPUT_DIR = "/content/local_images"
LOCAL_OUTPUT_DIR = "/content/local_results"
LOCAL_OUTPUT_ZIP = "/content/temp_output.zip"

def setup_env():
    print("Step 1: 드라이브 파일 압축 및 로컬 복사 중...")
    # 드라이브에서 직접 unzip하는 대신, 안정성을 위해 압축 후 복사
    os.system(f'zip -r -q {LOCAL_INPUT_ZIP} "{DRIVE_INPUT_DIR}"')

    print("Step 2: 로컬에 압축 푸는 중...")
    os.makedirs(LOCAL_INPUT_DIR, exist_ok=True)
    os.system(f'unzip -q {LOCAL_INPUT_ZIP} -d {LOCAL_INPUT_DIR}')
    os.makedirs(LOCAL_OUTPUT_DIR, exist_ok=True)

def process_images():
    print("Step 3: YOLO 작업 시작 (로컬 SSD 사용)...")
    model = YOLO(MODEL_PATH)

    # 로컬 경로에서 파일 탐색
    img_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
    image_paths = []
    for root, _, files in os.walk(LOCAL_INPUT_DIR):
        for f in files:
            if Path(f).suffix.lower() in img_exts:
                image_paths.append(Path(root) / f)

    print(f"총 {len(image_paths)}장 발견.")

    def crop_logic(img_path):
        # 상대 경로 유지하며 결과 경로 생성
        rel_path = img_path.relative_to(Path(LOCAL_INPUT_DIR).iterdir().__next__()) # 압축 풀린 구조 대응
        out_path = Path(LOCAL_OUTPUT_DIR) / rel_path
        out_path.parent.mkdir(parents=True, exist_ok=True)

        img = cv2.imread(str(img_path))
        if img is None: return

        results = model.predict(img, verbose=False) # fuse 옵션 제거
        if not results or len(results[0].boxes) == 0: return

        # 가장 큰 박스 찾기 [cite: 5, 6]
        best_box = max(results[0].boxes, key=lambda b: (b.xyxy[0][2]-b.xyxy[0][0]) * (b.xyxy[0][3]-b.xyxy[0][1]))
        x1, y1, x2, y2 = map(int, best_box.xyxy[0].tolist())

        crop = img[y1:y2, x1:x2]
        if RESIZE_TO_256 and crop.size != 0:
            crop = cv2.resize(crop, (256, 256), interpolation=cv2.INTER_AREA) [cite: 8]

        cv2.imwrite(str(out_path), crop)

    with ThreadPoolExecutor(max_workers=NUM_WORKERS) as ex:
        list(ex.map(crop_logic, image_paths))
    print("작업 완료.")

def cleanup_and_upload():
    print("Step 4: 결과물 압축 및 드라이브 전송...")
    os.system(f'zip -r -q {LOCAL_OUTPUT_ZIP} {LOCAL_OUTPUT_DIR}')
    shutil.copy(LOCAL_OUTPUT_ZIP, f"{DRIVE_OUTPUT_ROOT}.zip")

    print("Step 5: 로컬 캐시 삭제...")
    # /content/ 내의 임시 파일들 정리
    for p in [LOCAL_INPUT_ZIP, LOCAL_INPUT_DIR, LOCAL_OUTPUT_DIR, LOCAL_OUTPUT_ZIP]:
        if os.path.isfile(p): os.remove(p)
        elif os.path.isdir(p): shutil.rmtree(p)
    print("모든 작업이 종료되었습니다. 드라이브에서 .zip 파일을 확인하세요!")

if __name__ == "__main__":
    #setup_env()
    process_images()
    cleanup_and_upload()

Step 1: 드라이브 파일 압축 및 로컬 복사 중...
Step 2: 로컬에 압축 푸는 중...
Step 3: YOLO 작업 시작 (로컬 SSD 사용)...
총 13917장 발견.
Ultralytics 8.3.246 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Ultralytics 8.3.246 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Ultralytics 8.3.246 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Ultralytics 8.3.246 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Ultralytics 8.3.246 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Ultralytics 8.3.246 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Ultralytics 8.3.246 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Ultralytics 8.3.246 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 G

AttributeError: 'Conv' object has no attribute 'bn'

오류나서 step3,4만다시함

In [4]:
def run_process_and_upload():
    # 폴더 체크
    if not os.path.exists(LOCAL_INPUT_DIR) or not os.listdir(LOCAL_INPUT_DIR):
        print(f"❌ 에러: {LOCAL_INPUT_DIR} 폴더가 비어있습니다. Step 1, 2를 먼저 하셔야 해요!")
        return

    print("🚀 Step 3: YOLO 작업 시작...")
    model = YOLO(MODEL_PATH)
    os.makedirs(LOCAL_OUTPUT_DIR, exist_ok=True)

    # 이미지 파일 다 찾기
    img_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
    image_paths = []
    for root, _, files in os.walk(LOCAL_INPUT_DIR):
        for f in files:
            if Path(f).suffix.lower() in img_exts:
                image_paths.append(Path(root) / f)

    print(f"✅ 총 {len(image_paths)}장 발견. 처리 중...")

    def crop_logic(img_path):
        try:
            # 튼튼한 경로 계산: 파일명만 추출해서 결과 폴더에 저장
            # (만약 하위 폴더 구조까지 복잡하게 유지해야 하면 말씀해주세요!)
            out_path = Path(LOCAL_OUTPUT_DIR) / img_path.name

            img = cv2.imread(str(img_path))
            if img is None: return

            results = model.predict(img, verbose=False)
            if not results or len(results[0].boxes) == 0: return

            # 가장 큰 박스 선택 (박사님 TXT 파일 로직 반영)
            best_box = max(results[0].boxes, key=lambda b: (b.xyxy[0][2]-b.xyxy[0][0]) * (b.xyxy[0][3]-b.xyxy[0][1]))
            x1, y1, x2, y2 = map(int, best_box.xyxy[0].tolist())

            crop = img[y1:y2, x1:x2]
            if RESIZE_TO_256 and crop.size != 0:
                crop = cv2.resize(crop, (256, 256), interpolation=cv2.INTER_AREA)

            cv2.imwrite(str(out_path), crop)
        except Exception:
            pass # 에러 나도 멈추지 않고 다음 이미지로 진행

    # 병렬 실행 (NUM_WORKERS=4)
    with ThreadPoolExecutor(max_workers=NUM_WORKERS) as ex:
        ex.map(crop_logic, image_paths)

    print("✅ Step 4: 결과물 압축 및 드라이브 전송...")
    os.system(f'zip -r -q {LOCAL_OUTPUT_ZIP} {LOCAL_OUTPUT_DIR}')
    shutil.copy(LOCAL_OUTPUT_ZIP, f"{DRIVE_OUTPUT_ROOT}.zip")

    print(f"🎉 전송 완료! 드라이브의 {DRIVE_OUTPUT_ROOT}.zip 을 확인하세요.")

if __name__ == "__main__":
    run_process_and_upload()

🚀 Step 3: YOLO 작업 시작...
✅ 총 13917장 발견. 처리 중...
Ultralytics 8.3.246 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Ultralytics 8.3.246 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Ultralytics 8.3.246 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Ultralytics 8.3.246 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Ultralytics 8.3.246 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Ultralytics 8.3.246 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Ultralytics 8.3.246 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
Model summary (fused): 72 layers, 3,005,843 paramete

In [6]:
# 로컬파일삭제

import os
import shutil

# 삭제할 로컬 경로 리스트
targets = [
    "/content/local_images",    # 원본 압축 푼 폴더
    "/content/local_results",   # 결과물 저장 폴더
    "/content/temp_input.zip",  # 입력 압축 파일
    "/content/temp_output.zip"  # 결과 압축 파일
]

for path in targets:
    if os.path.isfile(path):
        os.remove(path)
        print(f"🗑️ 파일 삭제 완료: {path}")
    elif os.path.isdir(path):
        shutil.rmtree(path)
        print(f"🗑️ 폴더 삭제 완료: {path}")

print("\n✨ 모든 로컬 캐시가 정리되었습니다. 용량 확인해 보세요!")

🗑️ 폴더 삭제 완료: /content/local_images
🗑️ 폴더 삭제 완료: /content/local_results
🗑️ 파일 삭제 완료: /content/temp_input.zip
🗑️ 파일 삭제 완료: /content/temp_output.zip

✨ 모든 로컬 캐시가 정리되었습니다. 용량 확인해 보세요!
